# GDRN D3 - backend + PostGIS + k6 on Kaggle CPU

Private CPU job. Java 21, PostgreSQL/PostGIS and k6 run natively in the same session. Use a private offline runtime Dataset when Kaggle DNS is unavailable. No external backend, database credentials, tunnel, or public listener is required. Prepare with benchmark/kaggle_api.py.


## Reproducible job

The API preparer replaces only the tagged configuration cell. Source app commit, harness SHA-256 and source bundle SHA-256 are recorded. Smoke runs E07 write, E08 page and E08 radius; ladder adds page reads at 10/50/100 VUs. Each measurement resets this job-owned database; read measurements seed through E07.


In [ ]:
import json
JOB = {"configuration": {"profile": "smoke"}, "bundle": "", "bundleSha256": "", "appCommit": ""}


## Execute the isolated runtime

Online mode installs packages and builds on Kaggle. Offline mode uses a locally compiled JAR and checked Linux runtime binaries from a private Dataset; backend, database and generator still execute inside Kaggle CPU. Internet is disabled for offline jobs. Secrets are generated per job. Package builds are not a substitute for CI tests.


In [ ]:
import base64
import hashlib
import io
import json
import shutil
import sys
import tarfile
import tempfile
from pathlib import Path

def unpack(stream, destination):
    with tarfile.open(fileobj=stream, mode="r:gz") as archive:
        for member in archive:
            path = destination / member.name
            if not member.isfile() or not path.resolve().is_relative_to(destination.resolve()):
                raise RuntimeError("Invalid source/runtime archive member")
            path.parent.mkdir(parents=True, exist_ok=True)
            with archive.extractfile(member) as input_file, path.open("wb") as output_file:
                shutil.copyfileobj(input_file, output_file)
            path.chmod(member.mode & 0o777)

def checksum(path):
    digest=hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024*1024), b""): digest.update(block)
    return digest.hexdigest()

if not JOB.get("bundle"):
    raise RuntimeError("Prepare this notebook with benchmark/kaggle_api.py before uploading.")
payload=base64.b64decode(JOB["bundle"], validate=True)
if hashlib.sha256(payload).hexdigest()!=JOB["bundleSha256"]:
    raise RuntimeError("Source bundle checksum mismatch")
source_context=tempfile.TemporaryDirectory(prefix="gdrn-source-",dir="/tmp")
source=Path(source_context.name)
runtime_context=None
offline_runtime=None
try:
    unpack(io.BytesIO(payload),source)
    if JOB.get("runtimeDataset"):
        candidates=[]
        for path in Path("/kaggle/input").rglob("runtime-manifest.json"):
            metadata=json.loads(path.read_text())
            if metadata.get("kind")=="gdrn-offline-runtime" and metadata.get("archiveSha256")==JOB["runtimeArchiveSha256"]:
                candidates.append((path,metadata))
        if len(candidates)!=1: raise RuntimeError("Attach exactly one matching private offline runtime Dataset")
        manifest_path,metadata=candidates[0]
        if metadata["appCommit"]!=JOB["appCommit"]: raise RuntimeError("Offline JAR/source commit mismatch")
        runtime_context=tempfile.TemporaryDirectory(prefix="gdrn-offline-",dir="/tmp")
        runtime_root=Path(runtime_context.name)
        runtime_root.chmod(0o755)
        archive_path=manifest_path.parent/metadata["archive"]
        if archive_path.exists():
            if checksum(archive_path)!=JOB["runtimeArchiveSha256"]: raise RuntimeError("Offline archive checksum mismatch")
            with archive_path.open("rb") as handle: unpack(handle,runtime_root)
        else:
            # Kaggle may expand uploaded archives while ingesting a Dataset.
            roots=[manifest_path.parent/"runtime", manifest_path.parent/metadata["archive"].removesuffix(".tar.gz")/"runtime"]
            roots=[path for path in roots if path.is_dir()]
            if len(roots)!=1: raise RuntimeError("Cannot locate extracted runtime in Dataset")
            shutil.copytree(roots[0],runtime_root/"runtime")
            for path in (runtime_root/"runtime").rglob("*"):
                if path.is_file(): path.chmod(0o755)
        offline_runtime=runtime_root/"runtime"
        for name,digest in metadata["files"].items():
            path=offline_runtime/name
            if not path.resolve().is_relative_to(offline_runtime.resolve()) or checksum(path)!=digest:
                raise RuntimeError("Offline runtime file checksum mismatch")
    sys.path.insert(0,str(source/"benchmark"))
    from kaggle_server import NativeBenchmark
    output=Path("/kaggle/working/d3-results")
    benchmark=NativeBenchmark(source,output,JOB["configuration"],JOB["appCommit"],offline_runtime=offline_runtime)
    benchmark.manifest.update({key:JOB[key] for key in ("runtimeSha256","bundleSha256","runtimeDataset","runtimeArchiveSha256") if key in JOB})
    result=benchmark.execute()
finally:
    source_context.cleanup()
    if runtime_context: runtime_context.cleanup()


## Evidence and limits

Only benchmark evidence is stored under /kaggle/working/d3-results: manifests, seed records, raw k6 samples, sanitized summaries and process-group CPU/RSS. No database files or credentials are exported. The generator competes with the server for CPU; RSS sums can include shared pages. Downloaded evidence must be linked to the exact Kaggle notebook version. A successful preflight or green notebook without completed measurements does not prove D3 completion.


In [ ]:
print(json.dumps({"status": result["status"], "appCommit": result["appCommit"], "runs": len(result["runs"]), "cleanup": result["cleanup"]["status"], "output": str(output)}, indent=2))
